# Character Diffusion Language Model

Train a small character-level **discrete diffusion language model** from scratch. A bidirectional Transformer learns to recover text from randomly masked characters; generation begins with 256 masks and reveals them over 128 steps.

This is a self contained demonstration of absorbing-state D3PM on TinyStories. The default model has approximately 5 million parameters, and the notebook includes training, validation, unconditional generation, reconstruction, and prompt experiments.

## Run it

- **Colab:** choose a GPU runtime and run all cells. Checkpoints and the cached corpus go to `MyDrive/CharD3PM/tinystories_char_5m`.
- **Local Jupyter:** install `torch numpy matplotlib datasets huggingface_hub notebook`, then run all cells. Checkpoints and data go to `LOCAL_ROOT/RUN_NAME` (default `./character_d3pm_runs/tinystories_char_5m`). CPU works for small tests; full training is intended for a CUDA GPU.
- **Use your own text:** set `cfg['source'] = 'text_file'` and point `cfg['text_path']` to a UTF-8 file before the first run.

The notebook automatically resumes a valid checkpoint. Set `TRAIN=False` to load an existing run and use the demonstration cells without training. To move a Colab run locally, copy its complete run folder, including `data` and `checkpoints`, into `LOCAL_ROOT`. Change `RUN_NAME` for a separate experiment.

The default 135-minute limit applies to one training session; actual speed depends on the GPU. Saves occur every five minutes or 500 updates, on validation improvements, and at a normal stop. An interrupted session can lose work since its last completed save.


Reference: [D3PM](https://arxiv.org/abs/2107.03006) · [TinyStories](https://huggingface.co/datasets/roneneldan/TinyStories)

In [ ]:
import contextlib
import copy
import hashlib
import html
import importlib.util
import json
import math
import os
from pathlib import Path
import random
import re
import shutil
import subprocess
import sys
import tempfile
import time
import unicodedata
import uuid

import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from IPython.display import HTML, display

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Colab already supplies PyTorch. Only install the data reader if needed.
if IN_COLAB and importlib.util.find_spec('datasets') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'datasets>=3,<5'])

print('PyTorch:', torch.__version__)

In [ ]:
# Change RUN_NAME to start a separate experiment; an existing run auto-resumes.
RUN_NAME = 'tinystories_char_5m'
USE_GOOGLE_DRIVE = IN_COLAB        # Colab uses Drive; local Jupyter uses LOCAL_ROOT.
DRIVE_ROOT = '/content/drive/MyDrive/CharD3PM'
LOCAL_ROOT = './character_d3pm_runs'  # Local checkpoints and corpus; change as needed.
TRAIN = True                            # False: restore and run the generation cells.
SESSION_MINUTES = 135.0                 # Wall-clock limit for this invocation.
MICRO_BATCH = 64                        # Can be reduced on resume without changing the model.
MAX_UPDATES = None                      # Fresh runs only; None chooses from the GPU benchmark.

cfg = dict(
    seed=42,
    source='tinystories',               # Or 'text_file' for an uploaded UTF-8 corpus.
    text_path='/content/my_corpus.txt' if IN_COLAB else './my_corpus.txt',
    dataset_id='roneneldan/TinyStories',
    train_chars=20_000_000,
    val_chars=1_000_000,
    seq_len=256,
    timesteps=128,
    width=256,
    layers=6,
    heads=8,
    ff_multiplier=4,
    dropout=0.1,
    effective_batch=64,
    lr=3e-4,
    min_lr=3e-5,
    weight_decay=0.01,
    warmup_updates=200,
    grad_clip=1.0,
    eval_every=500,
    eval_batches=16,
    eval_batch=32,
    preview_every=1000,
    save_every_updates=500,
    save_every_seconds=300,
    keep_latest=3,
)
assert re.fullmatch(r'[A-Za-z0-9_-]+', RUN_NAME), 'Use letters, numbers, underscores or hyphens.'
assert SESSION_MINUTES > 0 and MICRO_BATCH > 0

## 1. Persistent storage and automatic recovery

Each save gets its own filename. A small JSON manifest records its size, SHA-256
checksum, update count, and validation score. The manifest is the completion marker.
Training reads cached data from local VM storage; Drive is used for durable copies.
Only tensor and ordinary Python state are serialized, allowing `weights_only=True`
when loading. Resume still validates the saved configuration and data fingerprint.

In [ ]:
FORMAT_VERSION = 1

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def publish_payload(folder, stem, payload, metadata=None):
    """Write locally, copy under a temporary name, verify, then publish the manifest."""
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    destination = folder / (stem + '.pt')
    partial = folder / (stem + '.pt.part')
    manifest_path = folder / (stem + '.json')
    with tempfile.TemporaryDirectory(prefix='chard3pm-save-') as temporary:
        local = Path(temporary) / 'payload.pt'
        torch.save(payload, local)
        checksum = sha256_file(local)
        size = local.stat().st_size
        shutil.copyfile(local, partial)
        if partial.stat().st_size != size or sha256_file(partial) != checksum:
            raise IOError('Checkpoint copy did not pass verification; previous snapshots are intact.')
        os.replace(partial, destination)
    manifest = dict(format_version=FORMAT_VERSION, file=destination.name,
                    bytes=size, sha256=checksum, created_ns=time.time_ns())
    manifest.update(metadata or {})
    manifest_partial = folder / (stem + '.json.part')
    manifest_partial.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    os.replace(manifest_partial, manifest_path)
    return manifest_path

def read_payload(manifest_path):
    manifest_path = Path(manifest_path)
    info = json.loads(manifest_path.read_text(encoding='utf-8'))
    if info['format_version'] != FORMAT_VERSION:
        raise ValueError('Unsupported artifact version.')
    filename = info['file']
    if Path(filename).name != filename or not filename.endswith('.pt'):
        raise ValueError('Invalid payload filename.')
    source = manifest_path.parent / filename
    if source.stat().st_size != info['bytes']:
        raise IOError('Size mismatch: ' + str(source))
    with tempfile.TemporaryDirectory(prefix='chard3pm-load-') as temporary:
        local = Path(temporary) / 'payload.pt'
        shutil.copyfile(source, local)
        if sha256_file(local) != info['sha256']:
            raise IOError('Checksum mismatch: ' + str(source))
        payload = torch.load(local, map_location='cpu', weights_only=True)
    return payload, info

class CheckpointStore:
    def __init__(self, run_dir, keep_latest=3):
        self.folder = Path(run_dir) / 'checkpoints'
        self.folder.mkdir(parents=True, exist_ok=True)
        self.keep_latest = keep_latest

    def records(self):
        records = []
        for path in self.folder.glob('step_*.json'):
            try:
                info = json.loads(path.read_text(encoding='utf-8'))
                assert isinstance(info['step'], int) and info['step'] >= 0
                assert isinstance(info['created_ns'], int)
                records.append((path, info))
            except Exception as error:
                print('Ignoring invalid manifest:', path.name, str(error))
        return sorted(records, key=lambda row: (row[1]['step'], row[1]['created_ns']), reverse=True)

    def load(self, which='latest'):
        assert which in ('latest', 'best')
        records = self.records()
        if which == 'best':
            records = sorted(
                [row for row in records if row[1].get('validation_bpc') is not None],
                key=lambda row: row[1]['validation_bpc'])
            if not records:
                print('No evaluated best snapshot yet; using latest.')
                return self.load('latest')
        for path, info in records:
            try:
                state, _ = read_payload(path)
                required = {'model', 'optimizer', 'scheduler', 'scaler', 'rng', 'batch_rng',
                            'cfg', 'vocab', 'data_fingerprint', 'step', 'total_steps', 'format_version'}
                if not required.issubset(state) or state['format_version'] != FORMAT_VERSION:
                    raise ValueError('Incomplete or unsupported training state.')
                if state['step'] != info['step']:
                    raise ValueError('Checkpoint/manifest step mismatch.')
                print(f"Loaded {which}: update {state['step']:,} — {path.with_suffix('.pt')}")
                return state
            except Exception as error:
                print('Recovery: skipping', path.name, '—', str(error))
        if list(self.folder.glob('step_*.json')):
            raise RuntimeError('Existing checkpoints could not be loaded. Repair the folder or choose a NEW RUN_NAME; refusing to restart silently.')
        print('No complete checkpoint found. This is a fresh run.')
        return None

    def save(self, state, validation_bpc=None):
        stem = f"step_{state['step']:09d}_{time.time_ns()}_{uuid.uuid4().hex[:6]}"
        path = publish_payload(self.folder, stem, state, dict(
            step=state['step'], validation_bpc=validation_bpc))
        records = self.records()
        retained = {p for p, _ in records[:self.keep_latest]}
        evaluated = [row for row in records if row[1].get('validation_bpc') is not None]
        if evaluated:
            retained.add(min(evaluated, key=lambda row: row[1]['validation_bpc'])[0])
        for old_path, info in records:
            if old_path not in retained:
                try:
                    # Remove completion marker first, so interrupted cleanup cannot advertise a missing file.
                    old_path.unlink()
                    (self.folder / info['file']).unlink(missing_ok=True)
                except OSError as error:
                    print('Old snapshot cleanup deferred:', error)
        print(f"Saved update {state['step']:,} to {path.with_suffix('.pt')}")
        return path

In [ ]:
if USE_GOOGLE_DRIVE:
    if not IN_COLAB:
        raise RuntimeError('Open this notebook in Colab, or set USE_GOOGLE_DRIVE=False for a local run.')
    from google.colab import drive
    drive.mount('/content/drive')
    root = Path(DRIVE_ROOT)
else:
    root = Path(LOCAL_ROOT).expanduser().resolve()

run_dir = root / RUN_NAME
run_dir.mkdir(parents=True, exist_ok=True)
# Detect a missing/unwritable mount before spending time training.
probe = run_dir / ('.write-test-' + uuid.uuid4().hex)
probe.write_text('ok', encoding='utf-8')
probe.unlink()
store = CheckpointStore(run_dir, cfg['keep_latest'])
resume_state = store.load('latest')
if resume_state is not None:
    cfg = copy.deepcopy(resume_state['cfg'])
    store.keep_latest = cfg['keep_latest']
    print('Restored saved architecture, data and optimizer configuration.')
    print('Planned updates:', resume_state['total_steps'], '| completed:', resume_state['step'])
else:
    print('A new run will use the configuration above.')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    props = torch.cuda.get_device_properties(device)
    print(f'GPU: {props.name} | {props.total_memory / 2**30:.1f} GiB')
else:
    print('CPU selected. Use a Colab GPU for the full training run.')
print('Persistent folder:', run_dir)

## 2. Prepare and cache the corpus

The first run streams a bounded subset, records the dataset revision, and saves the
encoded corpus and vocabulary to Drive. Later sessions reuse the same verified data.
Validation stories are selected first; matching training stories are excluded by their
normalized text hash. Buffered streaming shuffle selects from a bounded shuffled stream,
not a uniform sample of the entire dataset. The selected stories are shuffled again
before being joined with blank lines.

For another corpus, set `source='text_file'`, set `text_path`, and choose a new run name.
That mode reserves the last 10% of the normalized text for validation **before** drawing
windows. Upload a corpus large enough for both splits to contain 256 characters.

In [ ]:
def normalize_text(text):
    text = unicodedata.normalize('NFC', text)
    text = text.translate(str.maketrans({'“': '"', '”': '"', '‘': "'", '’': "'",
                                         '–': '-', '—': '-', '\u00a0': ' '}))
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    text = re.sub(r'[^\S\n]+', ' ', text)
    text = re.sub(r' *\n *', '\n', text)
    return re.sub(r'\n{3,}', '\n\n', text).strip()

def collect_stories(rows, target_chars, excluded, seed):
    stories, seen, total = [], set(excluded), 0
    for row in rows:
        story = normalize_text(row['text'])
        if not story:
            continue
        digest = hashlib.sha256(story.encode('utf-8')).hexdigest()
        if digest in seen:
            continue
        seen.add(digest)
        stories.append(story)
        total += len(story) + 2
        if len(stories) % 2000 == 0:
            print(f'Collected {len(stories):,} stories / {total:,} characters')
        if total >= target_chars:
            break
    if total < target_chars:
        raise ValueError(f'Dataset exhausted at {total:,} characters; requested {target_chars:,}.')
    random.Random(seed).shuffle(stories)
    return '\n\n'.join(stories), seen

def data_spec(config):
    keys = ('source', 'text_path', 'dataset_id', 'train_chars', 'val_chars', 'seed', 'seq_len')
    return {key: config[key] for key in keys}

def encode_text(text, vocab):
    stoi = {char: index for index, char in enumerate(vocab)}
    unknown = sum(char not in stoi for char in text)
    values = np.fromiter((stoi.get(char, stoi['?']) for char in text), dtype=np.int32, count=len(text))
    return torch.from_numpy(values), unknown

def build_data_payload(train_text, val_text, config, provenance):
    if min(len(train_text), len(val_text)) < config['seq_len']:
        raise ValueError('Both data splits must contain at least seq_len characters.')
    vocab = sorted(set(train_text) | {'?'})
    train_ids, _ = encode_text(train_text, vocab)
    val_ids, unknown = encode_text(val_text, vocab)
    identity = hashlib.sha256()
    identity.update(train_text.encode('utf-8'))
    identity.update(b'\x00VALIDATION\x00')
    identity.update(val_text.encode('utf-8'))
    return dict(train=train_ids, val=val_ids, vocab=vocab, spec=data_spec(config),
                fingerprint=identity.hexdigest(), provenance=provenance,
                val_unknown=unknown, train_text=train_text)

def prepare_data(config, directory):
    manifest = Path(directory) / 'data' / 'corpus.json'
    if manifest.exists():
        payload, _ = read_payload(manifest)
        if payload['spec'] != data_spec(config):
            raise ValueError('Cached data settings differ. Choose a new RUN_NAME for changed data.')
        print('Using verified cached corpus.')
        return payload

    if config['source'] == 'tinystories':
        from datasets import load_dataset
        from huggingface_hub import HfApi
        revision = HfApi().dataset_info(config['dataset_id']).sha
        def stream(split):
            return load_dataset(config['dataset_id'], revision=revision, split=split,
                                data_files={'train': 'data/train-*.parquet', 'validation': 'data/validation-*.parquet'},
                                streaming=True).shuffle(seed=config['seed'], buffer_size=1000)
        val_text, val_hashes = collect_stories(stream('validation'), config['val_chars'], set(), config['seed'] + 1)
        train_text, _ = collect_stories(stream('train'), config['train_chars'], val_hashes, config['seed'])
        provenance = dict(dataset=config['dataset_id'], revision=revision)
    elif config['source'] == 'text_file':
        text = normalize_text(Path(config['text_path']).read_text(encoding='utf-8'))
        split = int(len(text) * 0.9)
        train_text, val_text = text[:split], text[split:]
        provenance = dict(file=Path(config['text_path']).name,
                          sha256=hashlib.sha256(text.encode('utf-8')).hexdigest())
    else:
        raise ValueError('source must be tinystories or text_file.')
    payload = build_data_payload(train_text, val_text, config, provenance)
    publish_payload(manifest.parent, 'corpus', payload)
    return payload

def draw_batch(tokens, batch_size, length, generator, target_device):
    offsets = torch.randint(len(tokens) - length + 1, (batch_size,), generator=generator)
    indices = offsets[:, None] + torch.arange(length)[None, :]
    return tokens[indices].to(device=target_device, dtype=torch.long)

def decode_tokens(ids, vocab, mask_id):
    return ''.join('▮' if int(i) == mask_id else vocab[int(i)] for i in ids)

In [ ]:
if resume_state is not None and not (run_dir / 'data' / 'corpus.json').exists():
    raise RuntimeError('The checkpoint exists but its cached corpus is missing. Restore the data folder before resuming.')
data = prepare_data(cfg, run_dir)
vocab = data['vocab']
K = len(vocab)
MASK_ID = K
if resume_state is not None:
    if data['fingerprint'] != resume_state['data_fingerprint'] or vocab != resume_state['vocab']:
        raise ValueError('Checkpoint and corpus do not match; refusing to resume on different data.')
print(f"Train: {len(data['train']):,} characters | validation: {len(data['val']):,}")
print(f'Clean vocabulary: {K} | input vocabulary: {K + 1} | mask ID: {MASK_ID}')
print(f"Unseen validation characters mapped to '?': {data['val_unknown']:,} ({data['val_unknown'] / len(data['val']):.4%})")
print("Possible encoding artifact 'â€' in cached training text:", data['train_text'].count('â€'))
print('Training excerpt:\n' + data['train_text'][:400])

## 3. A small bidirectional Transformer

Character, learned position, and sinusoidal timestep embeddings are added together.
Six pre-LayerNorm Transformer blocks use bidirectional attention and GELU feed-forward
layers. The output head predicts only the clean character alphabet. All weights start
from random initialization. There is no pretrained model or teacher.

In [ ]:
class TimeEmbedding(nn.Module):
    def __init__(self, width):
        super().__init__()
        frequencies = torch.exp(-math.log(10000) * torch.arange(width // 2) / (width // 2 - 1))
        self.register_buffer('frequencies', frequencies, persistent=False)
        self.mlp = nn.Sequential(nn.Linear(width, width), nn.SiLU(), nn.Linear(width, width))

    def forward(self, t):
        angles = t.float()[:, None] * self.frequencies[None, :]
        return self.mlp(torch.cat([angles.sin(), angles.cos()], dim=-1))

class TransformerBlock(nn.Module):
    def __init__(self, width, heads, ff_multiplier, dropout):
        super().__init__()
        self.heads, self.dropout = heads, dropout
        self.norm1 = nn.LayerNorm(width)
        self.qkv = nn.Linear(width, width * 3)
        self.attn_out = nn.Linear(width, width)
        self.norm2 = nn.LayerNorm(width)
        self.ff_in = nn.Linear(width, width * ff_multiplier)
        self.ff_out = nn.Linear(width * ff_multiplier, width)

    def forward(self, x):
        batch, length, width = x.shape
        qkv = self.qkv(self.norm1(x)).reshape(batch, length, 3, self.heads, width // self.heads)
        q, k, v = qkv.permute(2, 0, 3, 1, 4).unbind(0)
        attention = F.scaled_dot_product_attention(
            q, k, v, dropout_p=self.dropout if self.training else 0.0, is_causal=False)
        attention = attention.transpose(1, 2).reshape(batch, length, width)
        x = x + F.dropout(self.attn_out(attention), p=self.dropout, training=self.training)
        hidden = F.gelu(self.ff_in(self.norm2(x)))
        hidden = F.dropout(hidden, p=self.dropout, training=self.training)
        return x + F.dropout(self.ff_out(hidden), p=self.dropout, training=self.training)

class CharacterD3PM(nn.Module):
    def __init__(self, clean_vocab_size, config):
        super().__init__()
        width = config['width']
        assert width >= 4 and width % 2 == 0 and width % config['heads'] == 0
        self.seq_len = config['seq_len']
        self.mask_id = clean_vocab_size
        self.token = nn.Embedding(clean_vocab_size + 1, width)
        self.position = nn.Embedding(self.seq_len, width)
        self.time = TimeEmbedding(width)
        self.dropout = config['dropout']
        self.blocks = nn.ModuleList([
            TransformerBlock(width, config['heads'], config['ff_multiplier'], config['dropout'])
            for _ in range(config['layers'])])
        self.norm = nn.LayerNorm(width)
        self.head = nn.Linear(width, clean_vocab_size)
        self.apply(self._initialize)
        # Keep the initial residual stream scale stable as layers accumulate.
        for block in self.blocks:
            nn.init.normal_(block.attn_out.weight, std=0.02 / math.sqrt(2 * config['layers']))
            nn.init.normal_(block.ff_out.weight, std=0.02 / math.sqrt(2 * config['layers']))

    @staticmethod
    def _initialize(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, std=0.02)
        if isinstance(module, nn.Linear) and module.bias is not None:
            nn.init.zeros_(module.bias)

    def forward(self, x_t, t):
        assert x_t.ndim == 2 and x_t.shape[1] <= self.seq_len
        positions = torch.arange(x_t.shape[1], device=x_t.device)
        x = self.token(x_t) + self.position(positions)[None] + self.time(t)[:, None]
        x = F.dropout(x, p=self.dropout, training=self.training)
        for block in self.blocks:
            x = block(x)
        return self.head(self.norm(x))

## 4. Forward diffusion, exact loss, and reverse sampling

For $T=128$, choose $\bar\alpha_t=1-t/T$. Independently mask each clean character with
probability $t/T$ to sample $x_t$ directly. At the endpoints, $x_0$ is clean and $x_T$
is entirely masked.

Let $M_i=1$ at masked positions. For a uniformly sampled timestep, the per-character
negative-ELBO estimator is

$$\mathcal L=\frac{T}{tL}\sum_{i=1}^{L}M_i\,[-\log p_\theta(x_{0,i}\mid x_t,t)].$$

To see the weight, the true reverse posterior of a masked position unmasks with
probability $1/t$. The model uses the same probability and supplies the clean character
distribution, so its per-position KL is masked cross-entropy divided by $t$. Multiplying
by $T$ accounts for sampling one of the $T$ terms. At $t=1$ this becomes the reconstruction
term. The terminal prior is exactly all masks, so its KL is zero. Visible positions are
copied in the reverse process and contribute zero. **Divide by sequence length, not by
the random number of masks.**

At inference, masked positions unmask independently with probability $1/t$, sampling
a clean character from the model. Visible positions stay fixed. This is progressive
unmasking: early commitments cannot be revised. Temperature 1 is the reference sampler;
other temperatures modify the generated distribution.

In [ ]:
def q_sample(x0, t, timesteps, mask_id, generator=None):
    probabilities = t.float()[:, None] / timesteps
    mask = torch.rand(x0.shape, device=x0.device, generator=generator) < probabilities
    return x0.masked_fill(mask, mask_id)

def weighted_d3pm_loss(logits, x0, xt, t, timesteps, mask_id):
    # FP32 log-softmax/cross-entropy even when Transformer matmuls use FP16.
    ce = F.cross_entropy(logits.float().transpose(1, 2), x0, reduction='none')
    masked = xt.eq(mask_id)
    per_example = (ce * masked).mean(dim=1) * (timesteps / t.float())
    return per_example.mean(), ce, masked

@torch.no_grad()
def reverse_step(model, xt, t, generator, temperature=1.0):
    assert t >= 1 and temperature > 0
    time_ids = torch.full((len(xt),), t, device=xt.device, dtype=torch.long)
    logits = model(xt, time_ids).float()
    unmask = xt.eq(model.mask_id) & (torch.rand(xt.shape, device=xt.device, generator=generator) < 1.0 / t)
    result = xt.clone()
    if unmask.any():
        probs = (logits[unmask] / temperature).softmax(dim=-1)
        result[unmask] = torch.multinomial(probs, 1, generator=generator).squeeze(-1)
    return result

@torch.no_grad()
def sample(model, config, target_device, batch_size=1, seed=123, temperature=1.0,
           template=None, snapshots=10):
    if temperature <= 0 or snapshots < 1:
        raise ValueError('temperature must be positive and snapshots must be at least 1.')
    generator = torch.Generator(device=target_device).manual_seed(seed)
    was_training = model.training
    model.eval()
    try:
        if template is None:
            canvas = torch.full((batch_size, config['seq_len']), model.mask_id,
                                dtype=torch.long, device=target_device)
        else:
            canvas = template.to(target_device).clone()
            if canvas.ndim == 1:
                canvas = canvas.unsqueeze(0)
            if canvas.ndim != 2 or not 0 < canvas.shape[1] <= config['seq_len']:
                raise ValueError('Template must contain between 1 and seq_len positions.')
            if ((canvas < 0) | (canvas > model.mask_id)).any():
                raise ValueError('Template contains an invalid token.')
        fixed = canvas.ne(model.mask_id)
        fixed_values = canvas.clone()
        frames = [(config['timesteps'], canvas.cpu().clone())]
        snapshot_steps = set(np.linspace(0, config['timesteps'], snapshots + 1, dtype=int).tolist())
        for t in range(config['timesteps'], 0, -1):
            with torch.autocast(device_type=target_device.type, dtype=torch.float16,
                                enabled=target_device.type == 'cuda'):
                canvas = reverse_step(model, canvas, t, generator, temperature)
            canvas[fixed] = fixed_values[fixed]
            if t - 1 in snapshot_steps:
                frames.append((t - 1, canvas.cpu().clone()))
        assert not canvas.eq(model.mask_id).any(), 'Reverse process left unresolved masks.'
        return canvas.cpu(), frames
    finally:
        model.train(was_training)

def show_trajectory(frames, alphabet, mask_id):
    rows = []
    for t, tokens in frames:
        text = html.escape(decode_tokens(tokens[0], alphabet, mask_id))
        text = text.replace('▮', '<span style="color:#999;background:#ececec">▮</span>')
        rows.append(f'<tr><td style="vertical-align:top;padding:8px">t={t}</td>'
                    f'<td><pre style="white-space:pre-wrap;overflow-wrap:anywhere;margin:8px">{text}</pre></td></tr>')
    display(HTML('<table style="width:100%;border-collapse:collapse">' + ''.join(rows) + '</table>'))

## 5. Optimization, validation, and reproducibility

AdamW uses a 200-update warmup and cosine decay. Gradient accumulation preserves an
effective batch of 64 if the microbatch is reduced. CUDA uses FP16 with gradient scaling;
the loss is computed in FP32. The learning-rate schedule advances only after successful
optimizer updates. A skipped update from the gradient scaler does not advance the counter.

Validation uses fixed windows and random seeds. The displayed **bound BPC** is a Monte
Carlo estimate of an upper bound on negative log-likelihood in bits per character,
not an exact likelihood or autoregressive perplexity. Separate masked accuracy/loss at
several noise levels helps expose reliance on nearly clean inputs. Evaluation and sampling
preserve the training random state.

In [ ]:
def capture_rng():
    numpy_state = np.random.get_state()
    return dict(python=random.getstate(),
                numpy=(numpy_state[0], numpy_state[1].tolist(), int(numpy_state[2]),
                       int(numpy_state[3]), float(numpy_state[4])),
                torch=torch.get_rng_state(),
                cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [])

def restore_rng(state):
    random.setstate(state['python'])
    name, keys, position, has_gaussian, cached = state['numpy']
    np.random.set_state((name, np.asarray(keys, dtype=np.uint32), position, has_gaussian, cached))
    torch.set_rng_state(state['torch'])
    if torch.cuda.is_available() and state['cuda']:
        if len(state['cuda']) == torch.cuda.device_count():
            torch.cuda.set_rng_state_all(state['cuda'])
        else:
            print('GPU count changed: resumed weights/optimizer; CUDA RNG cannot be reproduced exactly.')

@contextlib.contextmanager
def isolated_rng(seed):
    state = capture_rng()
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    try:
        yield
    finally:
        restore_rng(state)

def learning_rate_factor(update, total, warmup, minimum_ratio):
    warmup = min(warmup, max(1, total // 10))
    if update < warmup:
        return (update + 1) / warmup
    progress = min(1.0, (update - warmup) / max(1, total - warmup))
    return minimum_ratio + (1 - minimum_ratio) * 0.5 * (1 + math.cos(math.pi * progress))

class Trainer:
    def __init__(self, config, corpus, target_device, total_steps, micro_batch):
        self.cfg = copy.deepcopy(config)
        self.data, self.device = corpus, target_device
        self.total_steps = int(total_steps)
        self.micro_batch = int(micro_batch)
        if self.total_steps < 1 or self.cfg['effective_batch'] % self.micro_batch:
            raise ValueError('total_steps must be positive and microbatch must divide effective_batch.')
        self.model = CharacterD3PM(len(corpus['vocab']), config).to(target_device)
        matrices, vectors = [], []
        for parameter in self.model.parameters():
            (matrices if parameter.ndim >= 2 else vectors).append(parameter)
        self.optimizer = torch.optim.AdamW(
            [{'params': matrices, 'weight_decay': config['weight_decay']},
             {'params': vectors, 'weight_decay': 0.0}],
            lr=config['lr'], betas=(0.9, 0.999))
        self.scheduler = torch.optim.lr_scheduler.LambdaLR(
            self.optimizer, lambda update: learning_rate_factor(
                update, self.total_steps, config['warmup_updates'], config['min_lr'] / config['lr']))
        self.scaler = torch.amp.GradScaler('cuda', enabled=target_device.type == 'cuda')
        self.batch_rng = torch.Generator().manual_seed(config['seed'] + 11)
        self.step = 0
        self.best_bpc = float('inf')
        self.elapsed_seconds = 0.0
        self.history = []
        self.tokens_seen = 0

    def train_update(self):
        self.model.train()
        self.optimizer.zero_grad(set_to_none=True)
        accum = self.cfg['effective_batch'] // self.micro_batch
        total_loss = 0.0
        for _ in range(accum):
            x0 = draw_batch(self.data['train'], self.micro_batch, self.cfg['seq_len'], self.batch_rng, self.device)
            t = torch.randint(1, self.cfg['timesteps'] + 1, (len(x0),), device=self.device)
            xt = q_sample(x0, t, self.cfg['timesteps'], self.model.mask_id)
            with torch.autocast(device_type=self.device.type, dtype=torch.float16,
                                enabled=self.device.type == 'cuda'):
                logits = self.model(xt, t)
                loss, _, _ = weighted_d3pm_loss(logits, x0, xt, t, self.cfg['timesteps'], self.model.mask_id)
            if not torch.isfinite(loss):
                raise FloatingPointError('Non-finite loss. The previous complete checkpoint remains available.')
            self.scaler.scale(loss / accum).backward()
            total_loss += float(loss.detach()) / accum
        self.scaler.unscale_(self.optimizer)
        grad_norm = torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg['grad_clip'])
        if not self.scaler.is_enabled() and not torch.isfinite(grad_norm):
            raise FloatingPointError('Non-finite gradients.')
        old_scale = self.scaler.get_scale()
        self.scaler.step(self.optimizer)
        self.scaler.update()
        succeeded = self.scaler.get_scale() >= old_scale
        if succeeded:
            self.scheduler.step()
            self.step += 1
            self.tokens_seen += self.cfg['effective_batch'] * self.cfg['seq_len']
        self.optimizer.zero_grad(set_to_none=True)
        return total_loss, succeeded

    @torch.no_grad()
    def evaluate(self, batches=None, diagnostics=True):
        batches = self.cfg['eval_batches'] if batches is None else batches
        batch_size = min(self.cfg['eval_batch'], self.micro_batch)
        was_training = self.model.training
        self.model.eval()
        try:
            with isolated_rng(2026):
                generator = torch.Generator().manual_seed(2026)
                losses = []
                for _ in range(batches):
                    x0 = draw_batch(self.data['val'], batch_size, self.cfg['seq_len'], generator, self.device)
                    t = torch.randint(1, self.cfg['timesteps'] + 1, (batch_size,), device=self.device)
                    xt = q_sample(x0, t, self.cfg['timesteps'], self.model.mask_id)
                    with torch.autocast(device_type=self.device.type, dtype=torch.float16,
                                        enabled=self.device.type == 'cuda'):
                        loss, _, _ = weighted_d3pm_loss(self.model(xt, t), x0, xt, t,
                                                      self.cfg['timesteps'], self.model.mask_id)
                    losses.append(float(loss))
                result = dict(bound_bpc=float(np.mean(losses) / math.log(2)),
                              mc_standard_error_bpc=float(np.std(losses, ddof=1) / math.sqrt(len(losses)) / math.log(2)) if len(losses) > 1 else 0.0)
                if diagnostics:
                    levels = {}
                    for fraction in (0.0625, 0.25, 0.5, 0.75, 1.0):
                        t_value = max(1, round(fraction * self.cfg['timesteps']))
                        ce_sum, correct, count = 0.0, 0, 0
                        for _ in range(2):
                            x0 = draw_batch(self.data['val'], batch_size, self.cfg['seq_len'], generator, self.device)
                            t = torch.full((batch_size,), t_value, device=self.device, dtype=torch.long)
                            xt = q_sample(x0, t, self.cfg['timesteps'], self.model.mask_id)
                            with torch.autocast(device_type=self.device.type, dtype=torch.float16,
                                                enabled=self.device.type == 'cuda'):
                                logits = self.model(xt, t)
                            _, ce, mask = weighted_d3pm_loss(logits, x0, xt, t, self.cfg['timesteps'], self.model.mask_id)
                            ce_sum += float(ce[mask].sum())
                            correct += int((logits.argmax(-1).eq(x0) & mask).sum())
                            count += int(mask.sum())
                        levels[f'{t_value / self.cfg["timesteps"]:.1%} masked'] = {
                            'ce_nats': ce_sum / max(1, count), 'accuracy': correct / max(1, count)}
                    result['noise_levels'] = levels
                return result
        finally:
            self.model.train(was_training)

    def state_dict(self):
        return dict(format_version=FORMAT_VERSION, cfg=self.cfg, vocab=self.data['vocab'],
                    data_fingerprint=self.data['fingerprint'], model=self.model.state_dict(),
                    optimizer=self.optimizer.state_dict(), scheduler=self.scheduler.state_dict(),
                    scaler=self.scaler.state_dict(), rng=capture_rng(), batch_rng=self.batch_rng.get_state(),
                    step=self.step, total_steps=self.total_steps, best_bpc=self.best_bpc,
                    elapsed_seconds=self.elapsed_seconds, history=self.history, tokens_seen=self.tokens_seen)

    def load_state_dict(self, state):
        if (state['cfg'] != self.cfg or state['vocab'] != self.data['vocab'] or
                state['data_fingerprint'] != self.data['fingerprint'] or
                state['total_steps'] != self.total_steps):
            raise ValueError('Resume state does not match this model, schedule, vocabulary or corpus.')
        self.model.load_state_dict(state['model'])
        self.optimizer.load_state_dict(state['optimizer'])
        self.scheduler.load_state_dict(state['scheduler'])
        if state['scaler'] or not self.scaler.is_enabled():
            self.scaler.load_state_dict(state['scaler'])
        else:
            print('This checkpoint used full precision; starting a fresh CUDA gradient scaler.')
        self.step, self.best_bpc = state['step'], state['best_bpc']
        self.elapsed_seconds = state['elapsed_seconds']
        self.history = copy.deepcopy(state['history'])
        self.tokens_seen = state['tokens_seen']
        self.batch_rng.set_state(state['batch_rng'])
        restore_rng(state['rng'])

In [ ]:
def sanity_checks(target_device):
    with isolated_rng(7):
        toy_cfg = dict(seq_len=16, width=32, layers=1, heads=4, ff_multiplier=2, dropout=0.0)
        toy = CharacterD3PM(4, toy_cfg).to(target_device)
        x0 = (torch.arange(16, device=target_device) % 4)[None].repeat(4, 1)
        T = 8
        assert torch.equal(q_sample(x0, torch.zeros(4, device=target_device), T, 4), x0)
        assert q_sample(x0, torch.full((4,), T, device=target_device), T, 4).eq(4).all()
        xt = torch.full_like(x0, 4)
        t = torch.full((4,), T, dtype=torch.long, device=target_device)
        optimizer = torch.optim.AdamW(toy.parameters(), lr=0.01, weight_decay=0.0)
        for _ in range(100):
            optimizer.zero_grad(set_to_none=True)
            loss, _, _ = weighted_d3pm_loss(toy(xt, t), x0, xt, t, T, 4)
            assert torch.isfinite(loss)
            loss.backward()
            optimizer.step()
        assert float(loss.detach()) < 0.15, f'Tiny-batch overfit failed: {float(loss.detach()):.3f}'
        frozen = xt[0].clone()
        frozen[::3] = x0[0, ::3]
        completed, _ = sample(toy, dict(toy_cfg, timesteps=T), target_device, template=frozen)
        assert not completed.eq(4).any()
        assert torch.equal(completed[0, ::3], x0[0, ::3].cpu())
    print('Checks passed: corruption endpoints, finite gradients, tiny-batch overfit, fixed characters and zero final masks.')

def benchmark_training(config, corpus, target_device, requested_microbatch):
    microbatch = min(requested_microbatch, config['effective_batch'])
    if config['effective_batch'] % microbatch:
        raise ValueError('MICRO_BATCH must divide effective_batch (for example 64, 32, 16, 8).')
    while True:
        trial = None
        try:
            with isolated_rng(config['seed']):
                trial = Trainer(config, corpus, target_device, total_steps=1000, micro_batch=microbatch)
                for _ in range(3):
                    trial.train_update()
                if target_device.type == 'cuda':
                    torch.cuda.synchronize()
                start = time.perf_counter()
                for _ in range(10):
                    trial.train_update()
                if target_device.type == 'cuda':
                    torch.cuda.synchronize()
                seconds = (time.perf_counter() - start) / 10
            print(f'Benchmark: {seconds:.3f} seconds/update; microbatch={microbatch}; effective batch={config["effective_batch"]}.')
            return seconds, microbatch
        except torch.cuda.OutOfMemoryError:
            if target_device.type != 'cuda' or microbatch == 1:
                raise
            microbatch //= 2
            while config['effective_batch'] % microbatch:
                microbatch -= 1
            print('Reducing microbatch after GPU out-of-memory:', microbatch)
        finally:
            del trial
            if target_device.type == 'cuda':
                torch.cuda.empty_cache()

def train_for_session(trainer, checkpoint_store, session_minutes):
    if trainer.step >= trainer.total_steps:
        print('The saved training plan is complete. Continue to the demo cells.')
        return
    started = time.monotonic()
    elapsed_before = trainer.elapsed_seconds
    last_saved_time, last_saved_step = started, trainer.step
    inside_update = False
    skipped = 0

    def update_elapsed():
        trainer.elapsed_seconds = elapsed_before + time.monotonic() - started

    def save(score=None):
        nonlocal last_saved_time, last_saved_step
        update_elapsed()
        checkpoint_store.save(trainer.state_dict(), validation_bpc=score)
        last_saved_time, last_saved_step = time.monotonic(), trainer.step

    def validate():
        metrics = trainer.evaluate()
        print(f"Validation @ {trainer.step:,}: bound BPC={metrics['bound_bpc']:.3f} "
              f"(MC SE≈{metrics['mc_standard_error_bpc']:.3f})")
        update_elapsed()
        trainer.history.append(dict(step=trainer.step, elapsed_seconds=trainer.elapsed_seconds, **metrics))
        improved = metrics['bound_bpc'] < trainer.best_bpc
        if improved:
            trainer.best_bpc = metrics['bound_bpc']
            save(metrics['bound_bpc'])
        return improved

    try:
        if not trainer.history:
            validate()
        while trainer.step < trainer.total_steps:
            if time.monotonic() - started >= session_minutes * 60:
                print('Session time budget reached; saving before stopping.')
                break
            inside_update = True
            loss, succeeded = trainer.train_update()
            inside_update = False
            if not succeeded:
                skipped += 1
                if skipped >= 20:
                    raise FloatingPointError('20 consecutive AMP updates were skipped. Resume after checking numerical stability.')
                continue
            skipped = 0
            if trainer.step % 50 == 0 or trainer.step == 1:
                print(f"Update {trainer.step:,}/{trainer.total_steps:,} | loss {loss:.3f} | "
                      f"lr {trainer.optimizer.param_groups[0]['lr']:.2e} | "
                      f"session {(time.monotonic() - started)/60:.1f} min")
            if trainer.step % trainer.cfg['eval_every'] == 0:
                validate()
            # Check save deadlines before potentially expensive sampling previews.
            if (trainer.step - last_saved_step >= trainer.cfg['save_every_updates'] or
                    time.monotonic() - last_saved_time >= trainer.cfg['save_every_seconds']):
                save()
            if trainer.step % trainer.cfg['preview_every'] == 0:
                _, frames = sample(trainer.model, trainer.cfg, trainer.device, seed=123, snapshots=8)
                show_trajectory(frames, trainer.data['vocab'], trainer.model.mask_id)
        if not trainer.history or trainer.history[-1]['step'] != trainer.step:
            validate()
        if last_saved_step != trainer.step:
            save()
        print(f'Run stopped cleanly at update {trainer.step:,}; planned total {trainer.total_steps:,}.')
    except KeyboardInterrupt:
        if inside_update:
            print('Interrupted during an update. Its state is not published; restart from the last completed Drive checkpoint.')
        else:
            print('Interrupted between updates; saving completed progress.')
            save()
        raise
    except Exception:
        if not inside_update:
            try:
                save()
            except Exception as save_error:
                print('Emergency save failed:', save_error, 'Previous complete snapshots remain available.')
        raise

## 6. Verify, benchmark, initialize or resume

The short overfit test uses a separate toy model. The benchmark also uses disposable
weights and restores random states afterward. Neither changes the real model.

On a fresh run, the benchmark chooses a total update count with 10% of the training
allocation reserved for evaluation and saving. On resume, the original total and
learning-rate schedule are restored. `SESSION_MINUTES` remains a per-session stop limit.
GPU speed can change between sessions; unfinished plans can resume again. Exact bitwise
reproduction is only expected with the same hardware, software and microbatch settings.

The CPU path is for short checks: set `MAX_UPDATES` explicitly before training on CPU.

In [ ]:
sanity_checks(device)
needs_training = TRAIN and (resume_state is None or resume_state['step'] < resume_state['total_steps'])
if needs_training:
    if device.type == 'cpu' and MAX_UPDATES is None and resume_state is None:
        raise RuntimeError('Full training needs a GPU. Select a Colab GPU, or set a small MAX_UPDATES for a CPU check.')
    seconds_per_update, actual_microbatch = benchmark_training(cfg, data, device, MICRO_BATCH)
else:
    actual_microbatch = min(MICRO_BATCH, cfg['effective_batch'])

if resume_state is not None:
    planned_updates = resume_state['total_steps']
elif TRAIN:
    planned_updates = int(MAX_UPDATES) if MAX_UPDATES is not None else max(
        1, int(SESSION_MINUTES * 60 * 0.9 / seconds_per_update))
else:
    raise RuntimeError('No checkpoint exists to load. Set TRAIN=True for the first run.')

random.seed(cfg['seed'])
np.random.seed(cfg['seed'])
torch.manual_seed(cfg['seed'])
trainer = Trainer(cfg, data, device, planned_updates, actual_microbatch)
if resume_state is not None:
    trainer.load_state_dict(resume_state)
    del resume_state
    resume_state = None
else:
    # Initial snapshot also makes the selected plan recoverable before the first real update.
    store.save(trainer.state_dict())

print(f'Model parameters: {sum(p.numel() for p in trainer.model.parameters()):,}')
print(f'Updates: {trainer.step:,}/{trainer.total_steps:,} | characters processed: {trainer.tokens_seen:,}')
print(f'Automatic save interval: {cfg["save_every_seconds"]} seconds or {cfg["save_every_updates"]} completed updates.')

## 7. Train

Keep the notebook connected while it runs. Drive checkpoints protect completed saves;
an abrupt runtime termination can still lose the most recent unsaved interval. Deadlines
are checked between updates, and a save must finish before it is recoverable. A manual
interrupt inside an optimizer update also falls back to the last published snapshot.

After a disconnect, reopen the notebook and run it from the top with the **same Drive
folder and run name**. It loads automatically. Use one active runtime per run folder.

In [ ]:
if TRAIN:
    train_for_session(trainer, store, SESSION_MINUTES)
else:
    print('TRAIN=False: checkpoint restored for inspection and generation.')

## 8. Inspect learning and load the best model

`load_model_for_generation()` restores a saved model directly from Drive using the
vocabulary and architecture inside its checkpoint. It does not need the training data
or optimizer to generate. `which='latest'` selects the newest recoverable state;
`which='best'` selects the lowest recorded validation bound. Automatic training resume
always selects **latest**, preserving the most recent progress.

In [ ]:
def load_model_for_generation(checkpoint_store, target_device, which='best'):
    state = checkpoint_store.load(which)
    if state is None:
        raise FileNotFoundError('No saved model is available.')
    with isolated_rng(0):
        loaded_model = CharacterD3PM(len(state['vocab']), state['cfg']).to(target_device)
    loaded_model.load_state_dict(state['model'])
    loaded_model.eval()
    return loaded_model, state['cfg'], state['vocab'], state['step']

def save_now():
    """Optional manual save after a completed training cell."""
    return store.save(trainer.state_dict())

In [ ]:
if trainer.history:
    plt.figure(figsize=(7, 3))
    plt.plot([row['step'] for row in trainer.history], [row['bound_bpc'] for row in trainer.history], marker='o')
    plt.xlabel('Completed optimizer updates')
    plt.ylabel('Validation negative-ELBO bound (bits / character)')
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()
    print('Most recent validation diagnostics:')
    for level, values in trainer.history[-1].get('noise_levels', {}).items():
        print(f"  {level:>14} | masked CE {values['ce_nats']:.3f} nats | accuracy {values['accuracy']:.1%}")

demo_model, demo_cfg, demo_vocab, demo_step = load_model_for_generation(store, device, which='best')
print('Demo checkpoint update:', demo_step)

### Compare the best and latest checkpoints

The validation bound is estimated from a finite sample, so its lowest recorded value
may select an older checkpoint. These outputs use the same random seed and temperature.
If the latest checkpoint is visibly better, set `demo_model = latest_model` before
running the generation cells below. This comparison does not change saved checkpoints.

In [ ]:
latest_model, latest_cfg, latest_vocab, latest_step = load_model_for_generation(store, device, which='latest')
print(f'Best update: {demo_step:,} | latest update: {latest_step:,}')
if latest_step != demo_step:
    for label, candidate in (('BEST', demo_model), ('LATEST', latest_model)):
        candidate_text, _ = sample(candidate, demo_cfg, device, seed=91, snapshots=1)
        print(f'\n{label} SAMPLE:\n' + decode_tokens(candidate_text[0], demo_vocab, candidate.mask_id))
else:
    print('Best and latest refer to the same update.')

## 9. Unconditional generation — the main demonstration

Every position starts masked. The three independently sampled sequences use temperature
1.0 and a reproducible random stream. The first includes the complete unmasking trajectory. Judge ordinary sample
quality across seeds: spelling, word boundaries, punctuation, repetition and local
coherence. Early checkpoints may produce gibberish.

The overlap check below counts sampled 64-character blocks found verbatim in the
training corpus. It can flag copying, but it is not a complete memorization audit.

In [ ]:
generated, frames = sample(demo_model, demo_cfg, device, batch_size=3, seed=123, temperature=1.0)
show_trajectory(frames, demo_vocab, demo_model.mask_id)
for index, tokens in enumerate(generated):
    text = decode_tokens(tokens, demo_vocab, demo_model.mask_id)
    print(f'\nSAMPLE {index + 1}\n{text}')
    blocks = [text[start:start + 64] for start in range(0, max(0, len(text) - 63), 16)]
    matches = sum(block in data['train_text'] for block in blocks)
    print(f'64-character blocks seen verbatim in training: {matches}/{len(blocks)}')

## 10. Reconstruction, prefix completion and infilling

These are additional demonstrations. Reconstruction has access to genuine held-out
characters. Prefix completion and infilling use fixed-character clamping, an approximate
conditioning procedure for this unconditionally trained model. Their quality is reported
separately from all-mask generation. A fixed canvas always has 256 positions; there is no
learned end-of-sequence token, and a passage may end mid-sentence.

In [ ]:
# Reconstruct a held-out window with half its characters hidden.
clean = data['val'][:demo_cfg['seq_len']].long()
mask_rng = torch.Generator().manual_seed(17)
hidden = torch.rand(clean.shape, generator=mask_rng) < 0.5
template = clean.masked_fill(hidden, demo_model.mask_id)
reconstructed, _ = sample(demo_model, demo_cfg, device, template=template, seed=17)
print('ORIGINAL:\n' + decode_tokens(clean, demo_vocab, demo_model.mask_id))
print('\nCORRUPTED:\n' + decode_tokens(template, demo_vocab, demo_model.mask_id))
print('\nRECONSTRUCTED:\n' + decode_tokens(reconstructed[0], demo_vocab, demo_model.mask_id))
if hidden.any():
    print(f'Accuracy on the hidden characters: {float(reconstructed[0, hidden].eq(clean[hidden]).float().mean()):.1%}')

In [ ]:
# Edit the prefix and rerun this cell. Lengths are measured in characters.
PREFIX = 'Once upon a time, there was a little '
prefix_ids, unknown = encode_text(PREFIX, demo_vocab)
if len(prefix_ids) >= demo_cfg['seq_len']:
    raise ValueError('The prefix must leave at least one character to generate.')
if unknown:
    print(f"Mapped {unknown} unsupported prefix characters to '?'.")
prefix_canvas = torch.full((demo_cfg['seq_len'],), demo_model.mask_id, dtype=torch.long)
prefix_canvas[:len(prefix_ids)] = prefix_ids.long()
completion, frames = sample(demo_model, demo_cfg, device, template=prefix_canvas, seed=42)
show_trajectory(frames, demo_vocab, demo_model.mask_id)
print(decode_tokens(completion[0], demo_vocab, demo_model.mask_id))

In [ ]:
# Each ▮ represents exactly one missing character. Existing text stays fixed.
INFILL = 'The little ▮▮▮ went to the park.'
stoi = {char: index for index, char in enumerate(demo_vocab)}
infill_ids = torch.tensor([demo_model.mask_id if char == '▮' else stoi.get(char, stoi['?'])
                          for char in INFILL], dtype=torch.long)
filled, _ = sample(demo_model, demo_cfg, device, template=infill_ids, seed=9)
print('TEMPLATE:', INFILL)
print('FILLED:  ', decode_tokens(filled[0], demo_vocab, demo_model.mask_id))

### Continue later

Run from the top with the same `RUN_NAME`. The newest verified checkpoint resumes
automatically; completed plans go directly to the demo. Set `TRAIN=False` to load an
unfinished run without continuing training. To start a different dataset, architecture,
or training plan, choose a new run name. Existing run folders are not reset.

Optional commands after training:

```python
save_now()  # Extra full training snapshot to the same Drive folder.
model, config, alphabet, update = load_model_for_generation(store, device, which='latest')
```

**Interpretation:** the saved model is a small character language model with a
256-character context. This notebook verifies the implementation, but fluent output
and the achievable update count must be established by the actual GPU training run.